# 📊 InvestAI — Módulo NLP: Análisis de Sentimiento de Noticias Financieras

**Stack tecnológico:** NLTK VADER · GPT-4o (simulado) · Matplotlib · Pandas

**Empresas objetivo (Sector Minero):**

| Ticker         | Empresa                     |
|----------------|-----------------------------|
| FSM            | Fortuna Silver Mines        |
| VOLCABC1.LM    | Volcan Compañía Minera      |
| ABX.TO         | Barrick Gold Corporation    |
| BVN            | Buenaventura                |
| BHP            | BHP Group                   |

**Flujo del módulo:**
1. Instalación de NLTK y descarga del lexicón VADER
2. Dataset simulado de noticias financieras en inglés
3. Preprocesamiento de texto (limpieza y tokenización)
4. Análisis VADER: scores compound, pos, neg, neu
5. Clasificación: BULLISH (>0.05) | BEARISH (<-0.05) | NEUTRAL
6. Gauge chart de sentimiento consolidado
7. Exportación a `datos_nlp.json` (contrato con frontend)

> ⚙️ **Nota:** Los comentarios están en español. Los títulos y resúmenes de noticias están en inglés para compatibilidad óptima con el lexicón VADER.


## 📦 Celda 1 — Instalación de dependencias

In [ ]:
# Instalación de NLTK y librerías necesarias para el módulo NLP
# Ejecutar esta celda primero en Google Colab
!pip install nltk matplotlib pandas numpy --quiet

print("✅ Dependencias instaladas correctamente")


## 🔧 Celda 2 — Importaciones y descarga del lexicón VADER

In [ ]:
# Importación de librerías principales del módulo NLP
import nltk          # Natural Language Toolkit
import re            # Expresiones regulares para limpieza de texto
import json          # Exportación del archivo de datos para el frontend
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# ── Descarga de recursos NLTK requeridos ──────────────────────────────
# 'vader_lexicon': diccionario de sentimientos financieros/generales
# 'punkt':        tokenizador de oraciones
# 'stopwords':    palabras vacías (no usadas por VADER pero útiles en pipeline)
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt',         quiet=True)
nltk.download('stopwords',     quiet=True)

print("✅ NLTK cargado — Lexicón VADER disponible")
print(f"   Versión NLTK: {nltk.__version__}")


## 📰 Celda 3 — Dataset simulado de noticias financieras mineras

Se generan **15 noticias en inglés** (3 por empresa), con variedad de sentimientos.  
Los títulos y resúmenes están diseñados para producir señales claras en el lexicón VADER.


In [ ]:
# Dataset simulado de noticias financieras — Sector Minero Internacional
# 15 noticias: 3 por empresa (FSM, VOLCABC1.LM, ABX.TO, BVN, BHP)
# Texto en inglés: requerido por el lexicón VADER (diseñado para inglés)

noticias_raw = [

    # ══════════════════════════════════════════════════════════════════
    # FSM — Fortuna Silver Mines Inc.  (Canadá / México / W. África)
    # ══════════════════════════════════════════════════════════════════
    {
        "titulo": "Fortuna Silver Mines Reports Record Q1 2026 Silver Production; EPS Beats Consensus by 18%",
        "fuente": "Reuters Mining",
        "fecha":  "2026-06-12",
        "ticker": "FSM",
        "resumen": (
            "Fortuna Silver Mines posted record silver output of 2.4 million ounces in Q1 2026, "
            "surpassing analyst estimates by a wide margin. The company raised its full-year guidance "
            "and reported strong operating cash flows, driven by higher silver prices and remarkable "
            "efficiency gains at its Lindero and San Jose mines. Analysts at RBC upgraded the stock "
            "to Outperform citing exceptional operational execution."
        )
    },
    {
        "titulo": "Mexican Environmental Agency Revokes FSM Operating Permit at San Jose Mine; Shares Plunge 7%",
        "fuente": "Bloomberg Mining",
        "fecha":  "2026-06-09",
        "ticker": "FSM",
        "resumen": (
            "Mexico's SEMARNAT suspended Fortuna Silver's San Jose mine operations following "
            "allegations of soil contamination near community water sources. The company faces a "
            "mandatory 60-day environmental audit and potential fines. Shares fell sharply on the "
            "news as investors fear prolonged production shutdowns and mounting legal liability. "
            "The situation remains critical with no resolution timeline confirmed."
        )
    },
    {
        "titulo": "Fortuna Silver CEO to Present Capital Allocation Strategy at Denver Gold Forum 2026",
        "fuente": "PR Newswire",
        "fecha":  "2026-06-05",
        "ticker": "FSM",
        "resumen": (
            "Fortuna Silver Mines CEO Jorge Ganoza will address investors at the upcoming Denver "
            "Gold Forum, outlining the company's five-year capital allocation roadmap and "
            "exploration pipeline. The company will provide routine updates on its West Africa "
            "expansion and 2026-2027 production targets. Market reaction was muted as the "
            "presentation contains no material guidance changes."
        )
    },

    # ══════════════════════════════════════════════════════════════════
    # VOLCABC1.LM — Volcan Compañía Minera S.A.A.  (Perú — BVL)
    # ══════════════════════════════════════════════════════════════════
    {
        "titulo": "Volcan Mining Workers Strike Enters Day 12; Zinc Output at Cerro de Pasco Falls 40%",
        "fuente": "El Comercio Mining",
        "fecha":  "2026-06-11",
        "ticker": "VOLCABC1.LM",
        "resumen": (
            "A labor strike at Volcan's Cerro de Pasco operations entered its 12th day with no "
            "resolution in sight. Zinc production has dropped approximately 40% from planned output "
            "levels. The company and union representatives remain far apart on wage demands and safety "
            "protocols, raising serious concerns among investors about Q2 deliveries and contractual "
            "penalties. The dispute threatens to damage Volcan's relationships with key off-takers."
        )
    },
    {
        "titulo": "Volcan Reports Strongest Quarterly Earnings in Five Years as Zinc Surges Past $3,500 per Tonne",
        "fuente": "Gestión Finanzas",
        "fecha":  "2026-06-07",
        "ticker": "VOLCABC1.LM",
        "resumen": (
            "Volcan Compañía Minera delivered its best quarterly financial performance since 2021, "
            "driven by zinc prices surpassing $3,500 per tonne. The company reported EBITDA of "
            "$185 million, a remarkable 44% year-over-year increase. Management raised its annual "
            "zinc sales guidance and announced plans to accelerate the Chungar mine expansion, "
            "reinforcing investor confidence in long-term growth prospects."
        )
    },
    {
        "titulo": "Volcan Board Approves Dividend Reinvestment Plan; 2026 Capital Expenditure Guidance Maintained",
        "fuente": "BVL Noticias",
        "fecha":  "2026-06-03",
        "ticker": "VOLCABC1.LM",
        "resumen": (
            "Volcan's board of directors approved a dividend reinvestment program effective Q3 2026, "
            "allowing shareholders to reinvest dividends into new shares at a 3% discount. The company "
            "reaffirmed its $210 million capital expenditure plan for the year, with no major project "
            "additions or cancellations announced. Analysts noted the decision as routine corporate "
            "governance with neutral near-term market implications."
        )
    },

    # ══════════════════════════════════════════════════════════════════
    # ABX.TO — Barrick Gold Corporation  (Canadá — TSX / NYSE)
    # ══════════════════════════════════════════════════════════════════
    {
        "titulo": "Gold Breaks Historic $3,200 per Ounce Barrier; Barrick Gold Raises Full-Year Guidance by 8%",
        "fuente": "Financial Post",
        "fecha":  "2026-06-12",
        "ticker": "ABX.TO",
        "resumen": (
            "Gold surged to a historic $3,200 per ounce, propelled by US dollar weakness and "
            "rising geopolitical tensions. Barrick Gold capitalized on higher prices, revising its "
            "full-year production guidance upward by 8% to 4.6 million gold-equivalent ounces. "
            "Analysts at Scotiabank upgraded the stock to Outperform with a C$35 price target, "
            "citing exceptional cash flow generation and strengthened balance sheet position."
        )
    },
    {
        "titulo": "Tanzania Orders Suspension of Barrick North Mara Mine Over Alleged Cyanide Contamination",
        "fuente": "Africa Business Report",
        "fecha":  "2026-06-08",
        "ticker": "ABX.TO",
        "resumen": (
            "Tanzanian environmental authorities issued a suspension order for Barrick's North Mara "
            "gold mine following allegations of cyanide leakage into nearby rivers, threatening local "
            "communities. The mine contributes roughly 200,000 ounces annually. Barrick denied the "
            "accusations and filed for an emergency injunction, but significant uncertainty surrounding "
            "the dispute rattled investors and weighed heavily on the stock price."
        )
    },
    {
        "titulo": "Barrick Gold Signs MoU with Saudi Arabia to Develop New Gold Project in Arabian Shield",
        "fuente": "Reuters Commodities",
        "fecha":  "2026-06-10",
        "ticker": "ABX.TO",
        "resumen": (
            "Barrick Gold entered a memorandum of understanding with Saudi Arabia's Public Investment "
            "Fund to explore and develop gold deposits in the Arabian Shield region. The joint venture "
            "could unlock significant reserves with estimated potential of 2 million ounces at industry- "
            "leading grades. The partnership received strong positive market reaction, adding strategic "
            "diversification and opening a major growth opportunity for Barrick's portfolio."
        )
    },

    # ══════════════════════════════════════════════════════════════════
    # BVN — Compañía de Minas Buenaventura S.A.A.  (Perú — NYSE/BVL)
    # ══════════════════════════════════════════════════════════════════
    {
        "titulo": "Flash Floods Force Shutdown of BVN Uchucchacua Silver Mine; Production Outlook Slashed 12%",
        "fuente": "Peru Reports",
        "fecha":  "2026-06-11",
        "ticker": "BVN",
        "resumen": (
            "Severe flash flooding in Oyón province forced Buenaventura to halt operations at its "
            "Uchucchacua silver mine, one of the largest primary silver producers in Peru. The company "
            "drastically slashed its 2026 silver production outlook by 12%, sending shares down 5.4%. "
            "The recovery timeline remains uncertain pending critical safety and infrastructure "
            "assessments, adding pressure to an already challenging operational year."
        )
    },
    {
        "titulo": "Buenaventura Q2 2026 Production Beats All Estimates; Silver and Gold Output Surge 22% YoY",
        "fuente": "Semana Económica",
        "fecha":  "2026-06-04",
        "ticker": "BVN",
        "resumen": (
            "Buenaventura released outstanding preliminary Q2 2026 production data, showing combined "
            "silver and gold output up 22% year-over-year across all operating segments. The exceptional "
            "results exceeded consensus estimates significantly. The company attributed the gains to "
            "operational improvements at Yumpag, La Zanja, and Tambomayo mines, positioning BVN for "
            "what could be a record full-year performance in its history."
        )
    },
    {
        "titulo": "BVN Submits Trapiche Copper Project Environmental Study; Ministry Decision Expected in 90 Days",
        "fuente": "MEM Peru Official",
        "fecha":  "2026-06-01",
        "ticker": "BVN",
        "resumen": (
            "Buenaventura confirmed submission of the environmental impact study for its Trapiche "
            "copper project to Peru's Ministry of Energy and Mines. A regulatory decision is expected "
            "within 90 days. The project, if approved, would add significant copper production capacity "
            "from 2028 onward. Market reaction remained muted as investors await regulatory clarity "
            "before pricing in potential growth from the project."
        )
    },

    # ══════════════════════════════════════════════════════════════════
    # BHP — BHP Group Limited  (Australia — ASX / London / NYSE)
    # ══════════════════════════════════════════════════════════════════
    {
        "titulo": "BHP Posts Record Iron Ore Shipments in June Quarter; EV-Driven Copper Demand Hits 10-Year High",
        "fuente": "ASX Company News",
        "fecha":  "2026-06-12",
        "ticker": "BHP",
        "resumen": (
            "BHP Group reported record iron ore shipments from its Pilbara operations and a decade-high "
            "in copper demand driven by global electric vehicle adoption and energy transition spending. "
            "The company reaffirmed full-year copper production guidance of 1.9 million tonnes. Analysts "
            "at UBS raised their 12-month price target by 14% following the exceptional quarterly update, "
            "citing BHP's unmatched operational leverage to the energy transition."
        )
    },
    {
        "titulo": "BHP Raises Final Dividend 15% After Delivering Record Free Cash Flow of $18.4 Billion",
        "fuente": "Bloomberg Markets",
        "fecha":  "2026-06-06",
        "ticker": "BHP",
        "resumen": (
            "BHP announced a 15% increase in its final dividend to $1.35 per share after reporting "
            "full-year free cash flow of $18.4 billion, exceeding all analyst forecasts. CEO Mike Henry "
            "credited disciplined capital allocation and exceptionally favorable commodity prices. The "
            "share buyback program was extended by $4 billion, signaling strong management confidence "
            "in future cash generation and reinforcing BHP's status as a top-tier dividend payer."
        )
    },
    {
        "titulo": "China Steel Production Curbs Pressure Iron Ore Futures; BHP and Rio Tinto Hit Hardest",
        "fuente": "Reuters Commodities",
        "fecha":  "2026-06-02",
        "ticker": "BHP",
        "resumen": (
            "Iron ore futures fell more than 6% after China's government announced new production caps "
            "on steel mills in Hebei province to combat winter pollution levels. BHP and Rio Tinto saw "
            "their shares decline sharply as investors priced in lower demand for iron ore through "
            "year-end. Analysts warned the restrictions could extend into 2027, potentially eroding "
            "earnings and forcing dividend guidance revisions for major iron ore producers."
        )
    },
]

# Verificación del dataset
tickers_unicos = list(dict.fromkeys(n["ticker"] for n in noticias_raw))
print(f"📰 Noticias en el dataset  : {len(noticias_raw)}")
print(f"🏭 Empresas mineras cubiertas: {tickers_unicos}")
print(f"📅 Rango de fechas         : {min(n['fecha'] for n in noticias_raw)} → {max(n['fecha'] for n in noticias_raw)}")


## 🧹 Celda 4 — Preprocesamiento de texto: limpieza y tokenización

**Nota VADER:** A diferencia de otros modelos, VADER se beneficia de **no eliminar mayúsculas** ni signos de puntuación, ya que los usa como señales de énfasis emocional (e.g. `CRASH!` > `crash`).


In [ ]:
# Preprocesamiento de texto para análisis VADER
# VADER es sensible a: mayúsculas, signos !, ?, emojis y negaciones
# Por eso NO convertimos a minúsculas ni eliminamos puntuación clave

def preprocesar_texto(texto: str) -> str:
    """
    Limpieza básica que preserva las características lingüísticas
    que VADER usa para calcular la intensidad del sentimiento:
    - Mantiene mayúsculas (señal de énfasis)
    - Mantiene signos de puntuación (!, ?, ...)
    - Elimina caracteres especiales que generen ruido
    - Normaliza espacios múltiples
    """
    # Eliminar caracteres no-ASCII que no sean puntuación relevante
    texto = re.sub(r'[^\w\s\.\,\!\?\;\:\-\%\$\/\&]', ' ', texto)
    # Colapsar espacios múltiples en uno solo
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def tokenizar_texto(texto: str) -> list:
    """
    Tokenización simple por espacios.
    VADER opera directamente sobre el string completo,
    pero la lista de tokens puede usarse para análisis adicional.
    """
    return texto.split()

# ── Aplicar preprocesamiento a cada noticia ───────────────────────────
# Concatenamos título + resumen para dar más contexto al analizador VADER
for n in noticias_raw:
    texto_combinado         = n['titulo'] + '. ' + n['resumen']
    n['texto_procesado']    = preprocesar_texto(texto_combinado)
    n['tokens']             = tokenizar_texto(n['texto_procesado'])

# Mostrar ejemplo de preprocesamiento
print("✅ Preprocesamiento completado para todas las noticias\n")
print("=" * 65)
print("📌 EJEMPLO — FSM Noticia 1:")
print(f"   Tokens totales : {len(noticias_raw[0]['tokens'])}")
print(f"   Texto original :\n   {noticias_raw[0]['titulo'][:80]}...")
print(f"\n   Texto procesado:\n   {noticias_raw[0]['texto_procesado'][:80]}...")
print("=" * 65)


## 🤖 Celda 5 — Análisis de sentimiento VADER y clasificación

### Interpretación de scores VADER

| Score     | Rango     | Significado                              |
|-----------|-----------|------------------------------------------|
| `pos`     | [0, 1]    | Proporción de tokens positivos           |
| `neg`     | [0, 1]    | Proporción de tokens negativos           |
| `neu`     | [0, 1]    | Proporción de tokens neutros             |
| `compound`| [-1, 1]   | Score normalizado consolidado (el más importante) |

### Reglas de clasificación financiera

```
compound > 0.05   → BULLISH  (señal alcista)
compound < -0.05  → BEARISH  (señal bajista)
-0.05 ≤ c ≤ 0.05 → NEUTRAL
```


In [ ]:
# Inicializar el SentimentIntensityAnalyzer de VADER
sia = SentimentIntensityAnalyzer()

# ── Función de clasificación según reglas de inversión ────────────────
def clasificar_sentimiento(compound: float) -> str:
    """
    Clasifica el compound score de VADER en categorías de inversión.
    Umbrales estándar para análisis de sentimiento financiero.
    """
    if compound > 0.05:
        return "BULLISH"
    elif compound < -0.05:
        return "BEARISH"
    else:
        return "NEUTRAL"

# ── Simulación del consenso GPT-4o ────────────────────────────────────
def simular_gpt4o(sentimiento_vader: str, compound: float) -> str:
    """
    Simula el output del clasificador GPT-4o.
    En producción, esta función realizaría una llamada real a la API
    de OpenAI con el prompt de clasificación financiera.
    
    Introduce variabilidad realista:
    - Acuerdo total en noticias con señal fuerte (|compound| >= 0.4)
    - 12% de probabilidad de discrepancia en zona de incertidumbre
    """
    import random
    rng = random.Random(abs(int(compound * 10000)))  # Seed reproducible
    
    if abs(compound) < 0.3 and rng.random() < 0.12:
        # Zona gris: posible discrepancia entre VADER y LLM
        alternativas = [s for s in ["BULLISH", "NEUTRAL", "BEARISH"] if s != sentimiento_vader]
        return rng.choice(alternativas)
    return sentimiento_vader

# ── Procesar todas las noticias con VADER ─────────────────────────────
noticias_procesadas = []

for idx, n in enumerate(noticias_raw, start=1):
    # 1. Obtener los 4 scores de VADER
    scores = sia.polarity_scores(n['texto_procesado'])
    
    # 2. Clasificar según compound score
    sentimiento = clasificar_sentimiento(scores['compound'])
    
    # 3. Simular consenso GPT-4o
    gpt4o_sent = simular_gpt4o(sentimiento, scores['compound'])
    
    # 4. Construir objeto de noticia con estructura exacta del frontend
    noticia = {
        "id":          idx,
        "titulo":      n['titulo'],
        "fuente":      n['fuente'],
        "fecha":       n['fecha'],
        "resumen":     n['resumen'],
        "ticker":      n['ticker'],
        "sentimiento": sentimiento,
        "vader": {
            "pos":      round(scores['pos'],      3),
            "neg":      round(scores['neg'],      3),
            "neu":      round(scores['neu'],      3),
            "compound": round(scores['compound'], 3)
        },
        "gpt4o":    gpt4o_sent,
        "acuerdo":  (sentimiento == gpt4o_sent)
    }
    noticias_procesadas.append(noticia)

print("✅ Análisis VADER completado — Resultados:\n")

# ── Mostrar resultados en DataFrame ───────────────────────────────────
df = pd.DataFrame([{
    'ID':          n['id'],
    'Ticker':      n['ticker'],
    'Sentimiento': n['sentimiento'],
    'Compound':    n['vader']['compound'],
    'Pos':         n['vader']['pos'],
    'Neg':         n['vader']['neg'],
    'Neu':         n['vader']['neu'],
    'GPT-4o':      n['gpt4o'],
    'Acuerdo':     '✅' if n['acuerdo'] else '⚠️'
} for n in noticias_procesadas])

# Mostrar tabla completa
pd.set_option('display.max_colwidth', 14)
pd.set_option('display.width', 110)
print(df.to_string(index=False))


## 📊 Celda 6 — Resumen estadístico consolidado

In [ ]:
# ── Cálculo de estadísticas globales ─────────────────────────────────
total         = len(noticias_procesadas)
n_bullish     = sum(1 for n in noticias_procesadas if n['sentimiento'] == 'BULLISH')
n_bearish     = sum(1 for n in noticias_procesadas if n['sentimiento'] == 'BEARISH')
n_neutral     = sum(1 for n in noticias_procesadas if n['sentimiento'] == 'NEUTRAL')
compound_prom = sum(n['vader']['compound'] for n in noticias_procesadas) / total
acuerdo_pct   = sum(1 for n in noticias_procesadas if n['acuerdo']) / total * 100
sentimiento_agregado = clasificar_sentimiento(compound_prom)

# ── Estadísticas por empresa ──────────────────────────────────────────
print("=" * 62)
print("     RESUMEN GLOBAL — ANÁLISIS NLP SECTOR MINERO")
print("=" * 62)
print(f"  Total noticias analizadas  : {total}")
print(f"  ▲ BULLISH                  : {n_bullish:2d}  ({n_bullish/total*100:.0f}%)")
print(f"  ▼ BEARISH                  : {n_bearish:2d}  ({n_bearish/total*100:.0f}%)")
print(f"  ● NEUTRAL                  : {n_neutral:2d}  ({n_neutral/total*100:.0f}%)")
print(f"  Compound promedio VADER    : {compound_prom:+.4f}")
print(f"  Sentimiento del mercado    : {sentimiento_agregado}")
print(f"  Acuerdo VADER vs GPT-4o    : {acuerdo_pct:.0f}%")
print("=" * 62)

print("\n  DISTRIBUCIÓN POR EMPRESA:")
print(f"  {'Ticker':<16} {'Bull':>5} {'Bear':>5} {'Neu':>5}  {'Compound prom':>15}")
print(f"  {'-'*16} {'-'*5} {'-'*5} {'-'*5}  {'-'*15}")
for t in ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']:
    notas_t   = [n for n in noticias_procesadas if n['ticker'] == t]
    bull_t    = sum(1 for n in notas_t if n['sentimiento'] == 'BULLISH')
    bear_t    = sum(1 for n in notas_t if n['sentimiento'] == 'BEARISH')
    neu_t     = sum(1 for n in notas_t if n['sentimiento'] == 'NEUTRAL')
    comp_t    = sum(n['vader']['compound'] for n in notas_t) / len(notas_t)
    print(f"  {t:<16} {bull_t:>5} {bear_t:>5} {neu_t:>5}  {comp_t:>+15.4f}")
print("=" * 62)


## 📈 Celda 7 — Visualización: Gauge chart de sentimiento + distribución por empresa

Genera el velocímetro SVG-style con matplotlib y un gráfico de barras apiladas por ticker.


In [ ]:
# ── Paleta de colores coherente con investai_nlp.html ─────────────────
COLOR = {
    'azul':    '#1F3864',
    'dorado':  '#C5961A',
    'fondo':   '#F5F7FB',
    'bullish': '#26A69A',
    'bearish': '#EF5350',
    'neutral': '#78909C',
    'texto':   '#0D1B33',
    'suave':   '#6B7A99',
    'borde':   '#E2E8F3',
}

# Alias útiles para el gráfico
color_bull, color_bear, color_neu = COLOR['bullish'], COLOR['bearish'], COLOR['neutral']
color_sent = {'BULLISH': color_bull, 'BEARISH': color_bear, 'NEUTRAL': color_neu}
simbolo    = {'BULLISH': '▲', 'BEARISH': '▼', 'NEUTRAL': '●'}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.patch.set_facecolor(COLOR['fondo'])

# ══════════════════════════════════════════════════════════════════════
# PANEL IZQUIERDO — Velocímetro (Gauge chart)
# ══════════════════════════════════════════════════════════════════════
ax_g = axes[0]
ax_g.set_aspect('equal')
ax_g.set_xlim(-1.55, 1.55)
ax_g.set_ylim(-0.65, 1.45)
ax_g.axis('off')
ax_g.set_facecolor(COLOR['fondo'])

# ── Arcos de color por zona ───────────────────────────────────────────
lw_arco = 20

# Zona BEARISH (180° → 120°)
theta_b = np.linspace(np.pi, 2*np.pi/3, 100)
ax_g.plot(np.cos(theta_b), np.sin(theta_b),
          color=color_bear, linewidth=lw_arco, solid_capstyle='round', alpha=0.72, zorder=2)

# Zona NEUTRAL (120° → 60°)
theta_n = np.linspace(2*np.pi/3, np.pi/3, 100)
ax_g.plot(np.cos(theta_n), np.sin(theta_n),
          color=color_neu, linewidth=lw_arco, solid_capstyle='round', alpha=0.62, zorder=2)

# Zona BULLISH (60° → 0°)
theta_bu = np.linspace(np.pi/3, 0, 100)
ax_g.plot(np.cos(theta_bu), np.sin(theta_bu),
          color=color_bull, linewidth=lw_arco, solid_capstyle='round', alpha=0.72, zorder=2)

# ── Aguja de sentimiento ──────────────────────────────────────────────
# Mapear compound [-1, 1]  →  ángulo [180°, 0°]
angulo = np.pi * (1 - (compound_prom + 1) / 2)
tip_x  = 0.82 * np.cos(angulo)
tip_y  = 0.82 * np.sin(angulo)
ax_g.annotate(
    '',
    xy=(tip_x, tip_y), xytext=(0, 0),
    arrowprops=dict(arrowstyle='->', color=COLOR['azul'],
                    lw=3.8, mutation_scale=16)
)

# Centro del velocímetro
ax_g.add_patch(plt.Circle((0, 0), 0.11, color=COLOR['azul'],  zorder=5))
ax_g.add_patch(plt.Circle((0, 0), 0.06, color=COLOR['dorado'], zorder=6))

# ── Etiquetas de zona ─────────────────────────────────────────────────
ax_g.text(-1.35, -0.10, 'BEARISH', fontsize=9.5, fontweight='bold',
          color=color_bear, ha='center')
ax_g.text(0,     1.15,  'NEUTRAL', fontsize=9.5, fontweight='bold',
          color=color_neu, ha='center')
ax_g.text(1.35,  -0.10, 'BULLISH', fontsize=9.5, fontweight='bold',
          color=color_bull, ha='center')

# ── Valor numérico del compound ───────────────────────────────────────
ax_g.text(0, -0.32, f'{compound_prom:+.4f}', fontsize=19, fontweight='bold',
          color=COLOR['texto'], ha='center', va='center')
ax_g.text(0, -0.49, 'Compound Promedio VADER', fontsize=9,
          color=COLOR['suave'], ha='center', va='center')

# ── Badge de sentimiento agregado ─────────────────────────────────────
c_sent = color_sent[sentimiento_agregado]
rect_s = patches.FancyBboxPatch((-0.62, 0.74), 1.24, 0.30,
                                 boxstyle="round,pad=0.05",
                                 facecolor=c_sent, alpha=0.15,
                                 edgecolor=c_sent, linewidth=1.8, zorder=3)
ax_g.add_patch(rect_s)
ax_g.text(0, 0.89, f"{simbolo[sentimiento_agregado]} {sentimiento_agregado}",
          fontsize=13.5, fontweight='bold', color=c_sent,
          ha='center', va='center', zorder=4)

ax_g.set_title('Sentimiento Consolidado — Sector Minero',
               fontsize=12.5, fontweight='bold', color=COLOR['azul'], pad=12)

# ══════════════════════════════════════════════════════════════════════
# PANEL DERECHO — Barras agrupadas por empresa
# ══════════════════════════════════════════════════════════════════════
ax_b = axes[1]
ax_b.set_facecolor(COLOR['fondo'])

tickers = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
sents   = ['BULLISH', 'NEUTRAL', 'BEARISH']
datos   = {t: {s: 0 for s in sents} for t in tickers}
for n in noticias_procesadas:
    datos[n['ticker']][n['sentimiento']] += 1

x       = np.arange(len(tickers))
w       = 0.25
offsets = [-0.25, 0, 0.25]

for i, s in enumerate(sents):
    vals = [datos[t][s] for t in tickers]
    barras = ax_b.bar(x + offsets[i], vals, w,
                       label=s, color=color_sent[s],
                       alpha=0.86, edgecolor='white', linewidth=0.9,
                       zorder=3)
    for bar, val in zip(barras, vals):
        if val > 0:
            ax_b.text(bar.get_x() + bar.get_width() / 2,
                      bar.get_height() + 0.04, str(val),
                      ha='center', va='bottom', fontsize=10, fontweight='bold',
                      color=color_sent[s])

ax_b.set_xticks(x)
ax_b.set_xticklabels(tickers, fontsize=10, fontweight='600', color=COLOR['texto'])
ax_b.set_yticks([0, 1, 2, 3])
ax_b.set_ylim(0, 3.7)
ax_b.set_ylabel('Cantidad de noticias', fontsize=10, color=COLOR['suave'])
ax_b.set_title('Distribución de Sentimiento por Empresa Minera',
                fontsize=12.5, fontweight='bold', color=COLOR['azul'], pad=12)
ax_b.legend(loc='upper right', fontsize=9.5, framealpha=0.85,
            edgecolor=COLOR['borde'])
ax_b.grid(axis='y', color=COLOR['borde'], linewidth=0.9, alpha=0.9, zorder=0)
ax_b.spines[['top', 'right']].set_visible(False)
ax_b.spines['left'].set_color(COLOR['borde'])
ax_b.spines['bottom'].set_color(COLOR['borde'])

plt.suptitle(
    'InvestAI — NLP VADER: Análisis de Sentimiento de Noticias Mineras',
    fontsize=14, fontweight='bold', color=COLOR['azul'], y=1.03
)
plt.tight_layout(pad=2.2)
plt.savefig('gauge_sentimiento_minero.png', dpi=160,
            bbox_inches='tight', facecolor=COLOR['fondo'], edgecolor='none')
plt.show()
print("✅ Gráfico exportado como gauge_sentimiento_minero.png")


## 💾 Celda 8 — Exportación a `datos_nlp.json`

Genera el archivo JSON con la estructura **exacta** que consume el frontend `investai_nlp.html`.

Cada objeto del array tiene los campos requeridos por el contrato de datos:
`id · titulo · fuente · fecha · resumen · ticker · sentimiento · vader · gpt4o · acuerdo`


In [ ]:
# ── Exportación del JSON para el frontend ────────────────────────────
# Estructura validada contra investai_nlp.html (renderNoticia + actualizarPaneles)

RUTA_SALIDA = 'datos_nlp.json'

with open(RUTA_SALIDA, 'w', encoding='utf-8') as f:
    json.dump(noticias_procesadas, f, ensure_ascii=False, indent=2)

# ── Verificación del contrato de datos ───────────────────────────────
print("🔍 Verificando contrato de datos (investai_nlp.html)...\n")

CAMPOS_RAIZ  = ['id', 'titulo', 'fuente', 'fecha', 'resumen',
                'ticker', 'sentimiento', 'vader', 'gpt4o', 'acuerdo']
CAMPOS_VADER = ['pos', 'neg', 'neu', 'compound']

errores = 0
for n in noticias_procesadas:
    for campo in CAMPOS_RAIZ:
        if campo not in n:
            print(f"  ❌ Campo faltante '{campo}' en noticia id={n.get('id')}")
            errores += 1
    for vc in CAMPOS_VADER:
        if vc not in n.get('vader', {}):
            print(f"  ❌ vader.{vc} faltante en noticia id={n.get('id')}")
            errores += 1
    if n['sentimiento'] not in ('BULLISH', 'BEARISH', 'NEUTRAL'):
        print(f"  ❌ sentimiento inválido '{n['sentimiento']}' en id={n['id']}")
        errores += 1
    if n['gpt4o'] not in ('BULLISH', 'BEARISH', 'NEUTRAL'):
        print(f"  ❌ gpt4o inválido '{n['gpt4o']}' en id={n['id']}")
        errores += 1

if errores == 0:
    print("  ✅ Todos los campos del contrato están presentes y son válidos")
else:
    print(f"  ⚠️ Se detectaron {errores} errores en el contrato")

# ── Resumen final ─────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  ✅ {RUTA_SALIDA} exportado exitosamente")
print(f"     Registros  : {len(noticias_procesadas)}")
print(f"     Empresas   : FSM, VOLCABC1.LM, ABX.TO, BVN, BHP")
print(f"     Fechas     : 2026-06-01 → 2026-06-12")
print(f"{'='*55}")
print(f"\n📦 Ejemplo del primer registro exportado:")
print(json.dumps(noticias_procesadas[0], indent=2, ensure_ascii=False))


## 🔗 Celda 9 — Integración con el frontend `investai_nlp.html`

Una vez generado `datos_nlp.json`, para integrarlo con el frontend **reemplaza** el array hardcodeado en el HTML por una carga dinámica:

```html
<script>
  fetch('datos_nlp.json')
    .then(r => r.json())
    .then(data => {
      window.NOTICIAS = data;   // ← Variable global que consume renderFeed()
      renderFeed();             // ← Renderiza las tarjetas con los datos reales
    });
</script>
```

### Estructura de un objeto en `datos_nlp.json`
```json
{
  "id":          1,
  "titulo":      "Fortuna Silver Mines Reports Record Q1 2026 Silver Production...",
  "fuente":      "Reuters Mining",
  "fecha":       "2026-06-12",
  "resumen":     "Fortuna Silver Mines posted record silver output...",
  "ticker":      "FSM",
  "sentimiento": "BULLISH",
  "vader": {
    "pos":      0.312,
    "neg":      0.045,
    "neu":      0.643,
    "compound": 0.873
  },
  "gpt4o":   "BULLISH",
  "acuerdo": true
}
```

### Flujo de datos

```
[Colab Notebook]         [datos_nlp.json]         [investai_nlp.html]
noticias_raw ──VADER──▶  Array de objetos  ──▶  const NOTICIAS = [...]
                                                  ↓
                                              renderFeed()
                                              actualizarPaneles()
                                              gauge + chart + filtros
```
